The map visualises mean second-hand dwelling prices per square metre (in thousand HUF) across Hungary's statistical regions from 2007 to 2024, using data from the Hungarian Central Statistical Office (KSH) table 18.1.2.9. The data is extracted from two row types in the Excel file: the county seat rows (e.g. Budapest, Győr, Debrecen, Pécs, etc.) and the regional town average rows for each of Hungary's 7 NUTS2 regions. Each location is represented as a bubble placed at the city's geographic coordinates. Both the size and the colour of each bubble encode the same variable — the price per m² for that year — but they communicate it in complementary ways: the size makes differences immediately intuitive at a glance (a bubble twice as large means roughly twice the price), while the colour scale (green → yellow → red) allows precise cross-location comparison even when bubbles overlap or differ greatly in size. Using both together avoids the ambiguity that arises when relying on either dimension alone. The year slider animates the map through each year, so you can observe both the overall price surge after 2015 and the persistent and widening gap between Budapest and the rest of the country.

In [10]:
"""
Hungary Housing Prices – Bubble Map by County Seat City
========================================================
Uses the county seats + Budapest data from the Excel file.
Each city is shown as a coloured bubble whose size and colour
reflect the mean price per m² (thousand HUF) for that year.

Requirements:
    pip install pandas plotly openpyxl

Usage:
    python hungary_map.py

Output:
    hungary_housing_prices.html
"""

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ── 1. Parse Excel ─────────────────────────────────────────────────────────────

df_raw = pd.read_excel(
    "stadat-lak0025-18.1.2.9-en.xlsx",
    
    header=None,
)

years = [int(x) for x in df_raw.iloc[1, 2:].tolist()]

# Second-hand dwellings section
# Budapest = row 3 (capital); county seats = rows 7,11,15,19,23,27
# settlement type labels for the tooltip
city_rows = {
    # city name          : (row_index, region label)
    "Budapest":            (3,  "Budapest (capital)"),
    "Székesfehérvár":      (7,  "Central Transdanubia – county seat"),
    "Győr":                (11, "Western Transdanubia – county seat"),
    "Pécs":                (15, "Southern Transdanubia – county seat"),
    "Miskolc":             (19, "Northern Hungary – county seat"),
    "Debrecen":            (23, "Northern Great Plain – county seat"),
    "Kecskemét":           (27, "Southern Great Plain – county seat"),
}

# Approximate coordinates for each city
city_coords = {
    "Budapest":        (47.498,  19.040),
    "Székesfehérvár":  (47.190,  18.410),
    "Győr":            (47.683,  17.635),
    "Pécs":            (46.072,  18.233),
    "Miskolc":         (48.103,  20.778),
    "Debrecen":        (47.530,  21.629),
    "Kecskemét":       (46.896,  19.688),
}

records = []
for city, (row_idx, region) in city_rows.items():
    lat, lon = city_coords[city]
    for i, year in enumerate(years):
        val = df_raw.iloc[row_idx, i + 2]
        try:
            val = float(val)
        except (ValueError, TypeError):
            val = None
        records.append({
            "city":   city,
            "region": region,
            "year":   year,
            "price":  val,
            "lat":    lat,
            "lon":    lon,
        })

df = pd.DataFrame(records).sort_values("year").reset_index(drop=True)
df = df.dropna(subset=["price"])

# Also add towns for each region (for a richer map)
town_rows = {
    "Pest towns":                    (4,  "Pest – towns avg",          47.35, 19.50),
    "Central Transdanubia towns":    (8,  "Central Transdanubia towns", 47.35, 17.90),
    "Western Transdanubia towns":    (12, "Western Transdanubia towns", 47.20, 16.90),
    "Southern Transdanubia towns":   (16, "Southern Transdanubia towns",46.20, 17.60),
    "Northern Hungary towns":        (20, "Northern Hungary towns",     48.00, 20.20),
    "Northern Great Plain towns":    (24, "Northern Great Plain towns", 47.80, 21.50),
    "Southern Great Plain towns":    (28, "Southern Great Plain towns", 46.50, 19.80),
}

town_coords = {
    "Pest towns":                    (47.35, 19.50),
    "Central Transdanubia towns":    (47.35, 17.90),
    "Western Transdanubia towns":    (47.20, 16.90),
    "Southern Transdanubia towns":   (46.20, 17.60),
    "Northern Hungary towns":        (48.00, 20.20),
    "Northern Great Plain towns":    (47.80, 21.50),
    "Southern Great Plain towns":    (46.50, 19.80),
}

for label, (row_idx, region, lat, lon) in town_rows.items():
    for i, year in enumerate(years):
        val = df_raw.iloc[row_idx, i + 2]
        try:
            val = float(val)
        except (ValueError, TypeError):
            val = None
        records.append({
            "city":   label,
            "region": region,
            "year":   year,
            "price":  val,
            "lat":    lat,
            "lon":    lon,
        })

df_all = pd.DataFrame(records).sort_values("year").reset_index(drop=True)
df_all = df_all.dropna(subset=["price"])

# ── 2. Build animated bubble map ───────────────────────────────────────────────

price_min = df_all["price"].min()
price_max = df_all["price"].max()

fig = px.scatter_map(
    df_all,
    lat="lat",
    lon="lon",
    color="price",
    size="price",
    size_max=55,
    color_continuous_scale=["#2ecc71", "#f1c40f", "#e74c3c"],
    range_color=[price_min, price_max],
    animation_frame="year",
    hover_name="city",
    hover_data={"price": ":.0f", "region": True, "lat": False, "lon": False},
    labels={"price": "Price (thou. HUF/m²)"},
    title="Hungary – Second-hand Housing Price per m² (Thousand HUF)",
    center={"lat": 47.2, "lon": 19.3},
    zoom=6.0,
    map_style="carto-positron",
    height=650,
)

fig.update_layout(
    title_font_size=17,
    title_x=0.5,
    margin={"r": 0, "t": 60, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        title="Thou. HUF/m²",
        tickfont_size=11,
        len=0.6,
    ),
    sliders=[{"currentvalue": {"prefix": "Year: ", "font": {"size": 14}}}],
)

# ── 3. Save ────────────────────────────────────────────────────────────────────

fig.write_html("hungary_housing_prices.html")
print("Saved → hungary_housing_prices.html")
fig.show()

Saved → hungary_housing_prices.html


In [12]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load & parse data ──────────────────────────────────────────────────────────
df_raw = pd.read_excel(
    "stadat-lak0025-18.1.2.9-en.xlsx",
    
    header=None,
)

years = [int(x) for x in df_raw.iloc[1, 2:].tolist()]

# Second-hand dwellings – "together" row per region (Budapest = capital only)
region_rows = {
    "Budapest":              3,
    "Pest":                  6,
    "Central Transdanubia":  10,
    "Western Transdanubia":  14,
    "Southern Transdanubia": 18,
    "Northern Hungary":      22,
    "Northern Great Plain":  26,
    "Southern Great Plain":  30,
}

# Settlement-type breakdown rows (second-hand)
county_seat_rows = {
    "Pest": 4, "Central Transdanubia": 7, "Western Transdanubia": 11,
    "Southern Transdanubia": 15, "Northern Hungary": 19,
    "Northern Great Plain": 23, "Southern Great Plain": 27,
}
town_rows = {
    "Pest": 5, "Central Transdanubia": 8, "Western Transdanubia": 12,
    "Southern Transdanubia": 16, "Northern Hungary": 20,
    "Northern Great Plain": 24, "Southern Great Plain": 28,
}
village_rows = {
    "Pest": 6, "Central Transdanubia": 9, "Western Transdanubia": 13,
    "Southern Transdanubia": 17, "Northern Hungary": 21,
    "Northern Great Plain": 25, "Southern Great Plain": 29,
}

def parse_rows(row_dict, label_col="region", extra_col=None, extra_val=None):
    records = []
    for name, row_idx in row_dict.items():
        for i, year in enumerate(years):
            val = df_raw.iloc[row_idx, i + 2]
            try:
                val = float(val)
            except:
                val = None
            r = {label_col: name, "year": year, "price": val}
            if extra_col:
                r[extra_col] = extra_val
            records.append(r)
    return pd.DataFrame(records)

df = parse_rows(region_rows).dropna()
df = df.sort_values(["region", "year"]).reset_index(drop=True)

COLORS = {
    "Budapest":              "#e63946",
    "Pest":                  "#f4a261",
    "Central Transdanubia":  "#2a9d8f",
    "Western Transdanubia":  "#457b9d",
    "Southern Transdanubia": "#6a4c93",
    "Northern Hungary":      "#e9c46a",
    "Northern Great Plain":  "#264653",
    "Southern Great Plain":  "#606c38",
}

regions = list(COLORS.keys())
print("Data loaded:", df.shape, "rows")
print(df.head())


Data loaded: (144, 3) rows
     region  year  price
0  Budapest  2007  266.0
1  Budapest  2008  265.0
2  Budapest  2009  259.0
3  Budapest  2010  252.0
4  Budapest  2011  245.0


## Visualisation 1 – Line Chart: Price Evolution by Region
One line per region, showing the full 2007–2024 trajectory. Immediately reveals Budapest diverging sharply from the rest after 2015.

In [15]:

fig = go.Figure()

for region in regions:
    d = df[df["region"] == region]
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["price"],
        mode="lines+markers",
        name=region,
        line=dict(color=COLORS[region], width=2.5),
        marker=dict(size=5),
        hovertemplate="%{x}: %{y:.0f} thou. HUF/m²<extra>" + region + "</extra>",
    ))

fig.update_layout(
    title="Mean Second-hand Housing Price per m² by Region (2007–2024)",
    xaxis_title="Year",
    yaxis_title="Thousand HUF / m²",
    hovermode="x unified",
    legend=dict(orientation="v", x=1.01),
    template="plotly_white",
    height=500,
)
fig.show()

## Visualisation 2 – Small Multiples: One Chart per Region
Each region gets its own panel on the same y-axis scale, making it easy to compare shape without lines crossing. Ideal for academic publications.

In [16]:

from plotly.subplots import make_subplots

rows_n, cols_n = 2, 4
fig = make_subplots(rows=rows_n, cols=cols_n, subplot_titles=regions, shared_yaxes=True)

for idx, region in enumerate(regions):
    r, c = divmod(idx, cols_n)
    d = df[df["region"] == region]
    fig.add_trace(
        go.Scatter(
            x=d["year"], y=d["price"],
            mode="lines+markers",
            line=dict(color=COLORS[region], width=2),
            marker=dict(size=4),
            showlegend=False,
            name=region,
        ),
        row=r + 1, col=c + 1,
    )

fig.update_layout(
    title="Housing Price per m² – Small Multiples by Region",
    template="plotly_white",
    height=500,
)
fig.update_yaxes(title_text="Thou. HUF/m²", col=1)
fig.show()


## Visualisation 3 – Heatmap: Regions × Years
Rows = regions, columns = years, colour = price. The most compact overview: you can see at a glance which region/year cells are hot.

In [17]:

pivot = df.pivot(index="region", columns="year", values="price")
# Sort regions by 2024 price descending
pivot = pivot.loc[pivot[2024].sort_values(ascending=False).index]

fig = px.imshow(
    pivot,
    color_continuous_scale=["#2ecc71", "#f1c40f", "#e74c3c"],
    labels={"color": "Thou. HUF/m²", "x": "Year", "y": "Region"},
    title="Heatmap – Mean Housing Price per m² (Second-hand Dwellings)",
    aspect="auto",
    text_auto=".0f",
)
fig.update_layout(template="plotly_white", height=420, coloraxis_colorbar_title="Thou. HUF/m²")
fig.show()


## Visualisation 4 – Slope Chart: How Rankings Changed Over Time
Shows the rank (1 = most expensive) of each region for every year. Reveals whether poorer regions caught up or whether the hierarchy remained frozen.

In [18]:

df["rank"] = df.groupby("year")["price"].rank(ascending=False).astype(int)

fig = go.Figure()

for region in regions:
    d = df[df["region"] == region].sort_values("year")
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["rank"],
        mode="lines+markers+text",
        name=region,
        line=dict(color=COLORS[region], width=2),
        marker=dict(size=6),
        text=[region if yr in [2007, 2024] else "" for yr in d["year"]],
        textposition=["middle left" if yr == 2007 else "middle right" for yr in d["year"]],
        textfont=dict(size=10),
        hovertemplate="%{x}: rank %{y}<extra>" + region + "</extra>",
    ))

fig.update_yaxes(autorange="reversed", title="Rank (1 = most expensive)", tickmode="linear", tick0=1, dtick=1)
fig.update_xaxes(title="Year")
fig.update_layout(
    title="Regional Price Rankings Over Time (1 = Most Expensive)",
    template="plotly_white",
    height=500,
    showlegend=False,
)
fig.show()


## Visualisation 5 – Animated Bar Chart: Price Race Over Time
Horizontal bars ranked by price, animated year by year. Engaging for presentations and good for showing how the absolute gaps widen dramatically post-2015.

In [20]:

df_sorted = df.sort_values(["year", "price"])

fig = px.bar(
    df_sorted,
    x="price",
    y="region",
    orientation="h",
    animation_frame="year",
    color="region",
    color_discrete_map=COLORS,
    range_x=[0, df["price"].max() * 1.1],
    labels={"price": "Thou. HUF/m²", "region": ""},
    title="Housing Price per m² – Animated Bar Chart (2007–2024)",
    text="price",
)
fig.update_traces(texttemplate="%{x:.0f}", textposition="outside")
fig.update_layout(
    template="plotly_white",
    height=500,
    showlegend=False,
    sliders=[{"currentvalue": {"prefix": "Year: ", "font": {"size": 14}}}],
)
fig.show()


## Visualisation 6 – Index Chart: Cumulative Growth Since 2007
All regions normalised to 100 in 2007. Focuses on *growth rate* rather than absolute price. You may find that some cheaper regions grew proportionally faster than Budapest.

In [21]:

base_year = 2007
base = df[df["year"] == base_year].set_index("region")["price"]
df["index"] = df.apply(lambda row: (row["price"] / base[row["region"]]) * 100, axis=1)

fig = go.Figure()

for region in regions:
    d = df[df["region"] == region].sort_values("year")
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["index"],
        mode="lines+markers",
        name=region,
        line=dict(color=COLORS[region], width=2.5),
        marker=dict(size=5),
        hovertemplate="%{x}: %{y:.1f} (base 100 = 2007)<extra>" + region + "</extra>",
    ))

fig.add_hline(y=100, line_dash="dot", line_color="grey", line_width=1)
fig.update_layout(
    title="Housing Price Growth Index by Region (2007 = 100)",
    xaxis_title="Year",
    yaxis_title="Price index (2007 = 100)",
    hovermode="x unified",
    template="plotly_white",
    height=500,
    legend=dict(orientation="v", x=1.01),
)
fig.show()


# Cleaning the income data so it also 2007 and onwards

In [22]:

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Load income data ──────────────────────────────────────────────────────────
inc_raw = pd.read_excel(
    "stadat-gdp0035-21.1.1.35-en.xlsx",
   
    header=None,
)

# The file has two sections: "1990 = 100%" (rows 2–37) and "Previous year = 100%" (rows 38–73)
# We use the "1990 = 100%" cumulative index section (rows 3–36 → years 1991–2024)
# and filter to 2007 onward.

inc_cols = {
    "year":           0,
    "nom_wage_idx":   1,   # Index of avg net nominal wages (1990=100)
    "real_wage_idx":  2,   # Index of real wages (1990=100)
    "real_income_idx":3,   # Index of real income per capita (1990=100)
}

df_inc_raw = inc_raw.iloc[3:38, :4].copy()
df_inc_raw.columns = ["year", "nom_wage_idx", "real_wage_idx", "real_income_idx"]
df_inc_raw["year"] = pd.to_numeric(df_inc_raw["year"], errors="coerce")
for col in ["nom_wage_idx", "real_wage_idx", "real_income_idx"]:
    df_inc_raw[col] = pd.to_numeric(df_inc_raw[col], errors="coerce")

df_inc = df_inc_raw[(df_inc_raw["year"] >= 2007) & (df_inc_raw["year"] <= 2024)].dropna(subset=["year"]).copy()
df_inc["year"] = df_inc["year"].astype(int)
df_inc = df_inc.reset_index(drop=True)

# Also extract year-on-year % change section (rows 55–72 → years 2007–2024)
df_yoy_raw = inc_raw.iloc[55:73, :4].copy()
df_yoy_raw.columns = ["year", "nom_wage_yoy", "real_wage_yoy", "real_income_yoy"]
df_yoy_raw["year"] = pd.to_numeric(df_yoy_raw["year"], errors="coerce")
for col in ["nom_wage_yoy", "real_wage_yoy", "real_income_yoy"]:
    df_yoy_raw[col] = pd.to_numeric(df_yoy_raw[col], errors="coerce")
df_yoy = df_yoy_raw.dropna(subset=["year"]).copy()
df_yoy["year"] = df_yoy["year"].astype(int)
df_yoy = df_yoy.reset_index(drop=True)

# ── Load housing price data ───────────────────────────────────────────────────
house_raw = pd.read_excel(
    "stadat-lak0025-18.1.2.9-en.xlsx",
    
    header=None,
)

years_h = [int(x) for x in house_raw.iloc[1, 2:].tolist()]

region_rows = {
    "Budapest":              3,
    "Pest":                  6,
    "Central Transdanubia":  10,
    "Western Transdanubia":  14,
    "Southern Transdanubia": 18,
    "Northern Hungary":      22,
    "Northern Great Plain":  26,
    "Southern Great Plain":  30,
}

records = []
for region, row_idx in region_rows.items():
    for i, year in enumerate(years_h):
        val = house_raw.iloc[row_idx, i + 2]
        try:
            val = float(val)
        except:
            val = None
        records.append({"region": region, "year": year, "price": val})

df_house = pd.DataFrame(records).dropna()
df_house = df_house.sort_values(["region", "year"]).reset_index(drop=True)

# National average housing price (Country total row 35)
nat_price = {}
for i, year in enumerate(years_h):
    val = house_raw.iloc[35, i + 2]
    try:
        nat_price[year] = float(val)
    except:
        nat_price[year] = None
df_nat_price = pd.DataFrame(list(nat_price.items()), columns=["year", "nat_price"]).dropna()

# ── Rebase housing prices to 2007 = 100 for comparison ───────────────────────
base_2007_price = {}
for region in region_rows:
    bp = df_house[(df_house["region"] == region) & (df_house["year"] == 2007)]["price"].values
    base_2007_price[region] = bp[0] if len(bp) else None

df_house["price_idx"] = df_house.apply(
    lambda r: (r["price"] / base_2007_price[r["region"]]) * 100
    if base_2007_price.get(r["region"]) else None, axis=1
)

# Rebase national average to 2007=100
base_nat = df_nat_price[df_nat_price["year"] == 2007]["nat_price"].values[0]
df_nat_price["nat_price_idx"] = (df_nat_price["nat_price"] / base_nat) * 100

# Rebase income indices to 2007=100
base_real_income = df_inc[df_inc["year"] == 2007]["real_income_idx"].values[0]
base_real_wage   = df_inc[df_inc["year"] == 2007]["real_wage_idx"].values[0]
base_nom_wage    = df_inc[df_inc["year"] == 2007]["nom_wage_idx"].values[0]

df_inc["real_income_2007"] = (df_inc["real_income_idx"] / base_real_income) * 100
df_inc["real_wage_2007"]   = (df_inc["real_wage_idx"]   / base_real_wage)   * 100
df_inc["nom_wage_2007"]    = (df_inc["nom_wage_idx"]    / base_nom_wage)    * 100

REGION_COLORS = {
    "Budapest":              "#e63946",
    "Pest":                  "#f4a261",
    "Central Transdanubia":  "#2a9d8f",
    "Western Transdanubia":  "#457b9d",
    "Southern Transdanubia": "#6a4c93",
    "Northern Hungary":      "#e9c46a",
    "Northern Great Plain":  "#264653",
    "Southern Great Plain":  "#606c38",
}
regions = list(REGION_COLORS.keys())

print("Income data (2007–2024):")
print(df_inc[["year","real_income_2007","real_wage_2007","nom_wage_2007"]].to_string(index=False))
print("\nHousing data rows:", len(df_house))


Income data (2007–2024):
 year  real_income_2007  real_wage_2007  nom_wage_2007
 2007        100.000000      100.000000     100.000000
 2008         97.989510      100.800000     107.000000
 2009         93.968531       98.481600     108.926000
 2010         93.706294      100.254269     116.332968
 2011         97.552448      102.773892     123.776983
 2012         95.541958       99.249747     126.376897
 2013         97.639860      102.364108     132.569259
 2014        101.748252      105.642382     136.548181
 2015        105.594406      110.313922     142.415555
 2016        110.227273      118.438936     153.525817
 2017        117.132867      130.583554     173.330647
 2018        126.748252      141.406525     192.919029
 2019        135.489510      153.304366     216.262231
 2020        134.440559      162.060375     236.158357
 2021        145.716783      167.148855     255.995658
 2022        150.961538      172.550172     302.586868
 2023        153.059441      167.268024 

## Visualisation 1 – Income Indices Over Time (2007 = 100)
All three series rebased to 100 in 2007 so they start at the same point and divergence is immediately visible. Nominal wages grow fastest (they include inflation); real wages and real income per capita grow more modestly, with real income dipping during the 2009 crisis and the 2022–23 inflationary shock.

Nominal wages are the raw number on your payslip in Hungarian forints. They grow whenever employers raise salaries, but they say nothing about what that money can actually buy — if prices rise faster than your salary, you are worse off even though the number went up.
Real wages adjust nominal wages for inflation (using the consumer price index). This tells you whether your purchasing power — what your salary can actually buy in shops — has increased or decreased. When inflation is high, like in Hungary in 2022–2023, real wages can fall even while nominal wages are rising, because prices outrun salary increases. This is exactly the dip you see in Viz 2 around that period.
Real income per capita is broader than real wages. It includes not just employment wages but also pensions, social transfers, welfare benefits, self-employment income, and other household income sources — all adjusted for inflation and divided by the total population (not just workers). It can diverge from real wages because of government policy: for example, pension increases, family subsidies, or minimum wage hikes can push real income per capita up even in years when average real wages stagnate, and vice versa. It also captures the effect of people moving in or out of employment.
In short: nominal wages tell you the number, real wages tell you the purchasing power of workers, and real income per capita tells you the purchasing power of the whole population. For housing affordability, real income per capita is arguably the most honest benchmark, since housing is purchased by households — not just wage earners.

In [24]:

fig = go.Figure()

series = [
    ("nom_wage_2007",    "Nominal wage index (2007=100)",     "#e63946", "solid", 2.5),
    ("real_wage_2007",   "Real wage index (2007=100)",        "#2a9d8f", "solid", 2.5),
    ("real_income_2007", "Real income per capita (2007=100)", "#457b9d", "dash",  2.5),
]

for col, label, color, dash, width in series:
    fig.add_trace(go.Scatter(
        x=df_inc["year"], y=df_inc[col],
        mode="lines+markers",
        name=label,
        line=dict(color=color, width=width, dash=dash),
        marker=dict(size=5),
        hovertemplate="%{x}: %{y:.1f}<extra>" + label + "</extra>",
    ))

fig.add_hline(y=100, line_dash="dot", line_color="lightgrey", line_width=1,
              annotation_text="2007 baseline", annotation_position="bottom right")

fig.update_layout(
    title="Hungary – Income & Wage Indices Rebased to 2007 = 100",
    xaxis_title="Year",
    yaxis_title="Index (2007 = 100)",
    template="plotly_white",
    height=480,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.2),
)
fig.show()


## Visualisation 2 – Year-on-Year Change in Real Income & Real Wages (%)
Annual percentage change relative to the previous year. Values below 100 indicate a real decline. The 2009 crisis dip and the 2022–2023 inflationary squeeze are clearly visible.

In [25]:

# Convert to growth rate: index - 100 gives the % change
df_yoy["real_wage_growth"]   = df_yoy["real_wage_yoy"]   - 100
df_yoy["real_income_growth"] = df_yoy["real_income_yoy"] - 100

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=["Real Wage – Annual % Change",
                                    "Real Income per Capita – Annual % Change"])

for row, (col, color) in enumerate([("real_wage_growth","#2a9d8f"),
                                     ("real_income_growth","#457b9d")], start=1):
    fig.add_trace(go.Bar(
        x=df_yoy["year"],
        y=df_yoy[col],
        marker_color=[color if v >= 0 else "#e63946" for v in df_yoy[col]],
        showlegend=False,
        hovertemplate="%{x}: %{y:+.1f}%<extra></extra>",
    ), row=row, col=1)
    fig.add_hline(y=0, line_dash="dot", line_color="black", line_width=1, row=row, col=1)

fig.update_layout(
    title="Hungary – Year-on-Year Real Income & Wage Growth (%)",
    template="plotly_white",
    height=520,
)
fig.update_yaxes(title_text="% change", col=1)
fig.show()


## Visualisation 4 – Housing Prices vs Income: Growth Index (2007 = 100) ★ KEY CHART
Everything rebased to 100 in 2007. Income series (national) are plotted as thick lines; regional housing price indices as thinner coloured lines. **If a housing line rises faster than the income lines, housing has become less affordable in that region.**

In [27]:

fig = go.Figure()

# Income traces (thick, dashed variants)
income_series = [
    ("real_income_2007", "Real income per capita",  "#000000", "dot",   4),
    ("real_wage_2007",   "Real wage index",          "#333333", "dash",  3),
    ("nom_wage_2007",    "Nominal wage index",       "#888888", "solid", 2),
]
for col, label, color, dash, width in income_series:
    fig.add_trace(go.Scatter(
        x=df_inc["year"], y=df_inc[col],
        mode="lines",
        name=label,
        line=dict(color=color, width=width, dash=dash),
        hovertemplate="%{x}: %{y:.1f}<extra>" + label + "</extra>",
    ))

# Housing price index traces (coloured, thinner)
for region in regions:
    d = df_house[df_house["region"] == region].sort_values("year")
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["price_idx"],
        mode="lines+markers",
        name=region + " (housing)",
        line=dict(color=REGION_COLORS[region], width=1.8),
        marker=dict(size=4),
        hovertemplate="%{x}: %{y:.1f}<extra>" + region + " housing</extra>",
    ))

fig.add_hline(y=100, line_dash="dot", line_color="lightgrey", line_width=1)
fig.update_layout(
    title="Housing Price Growth vs Income Growth – All Rebased to 2007 = 100",
    xaxis_title="Year",
    yaxis_title="Index (2007 = 100)",
    template="plotly_white",
    height=560,
    hovermode="x unified",
    legend=dict(orientation="v", x=1.01, font=dict(size=11)),
)
fig.show()


## Visualisation 5 – Affordability Ratio: Housing Price Index / Real Income Index
If this ratio rises above 1.0 (the 2007 baseline), housing prices have outpaced real income growth — meaning homes have become **less affordable**. A value of 1.5 means housing costs 50% more relative to income than in 2007.

The affordability ratio is built in two steps.
Step 1 — Rebase both series to 2007 = 100. The housing price for each region is divided by its own 2007 value and multiplied by 100, giving a housing price index. The same is done for real income per capita nationally. This puts both series on the same neutral starting point regardless of their original units (thousand HUF/m² for housing, a dimensionless index for income), so their growth rates become directly comparable.
Step 2 — Divide the housing index by the income index. For each region and each year:

affordability ratio = (housing price index) / (real income per capita index)

In 2007 this ratio equals exactly 1.0 for every region by construction. In subsequent years, if housing prices grow faster than real income, the numerator rises faster than the denominator and the ratio climbs above 1.0. A ratio of 1.5 means that housing has become 50% more expensive relative to income compared to 2007 — i.e. a household would need to spend 50% more of their income on housing than they did in 2007 for the same square meterage. A ratio below 1.0 would mean housing became more affordable relative to income, which you will not find much of in this dataset.
What this measure does not capture. This is a relative affordability measure, not an absolute one. It does not tell you how many months of income are needed to buy a flat — for that you would need price-to-income ratio data at the transaction level. It also uses a national income index applied uniformly to all regions, because the KSH income data is not broken down regionally in this dataset. This means the affordability ratio reflects regional housing price divergence accurately, but assumes all regions had the same income growth — which is a simplification, since Budapest wages likely grew faster than the national average, making Budapest's true affordability situation slightly less severe than the ratio suggests.

In [28]:

# Merge housing index with income index on year
df_aff = df_house[["region","year","price_idx"]].merge(
    df_inc[["year","real_income_2007"]], on="year"
)
df_aff["afford_ratio"] = df_aff["price_idx"] / df_aff["real_income_2007"]

fig = go.Figure()
for region in regions:
    d = df_aff[df_aff["region"] == region].sort_values("year")
    fig.add_trace(go.Scatter(
        x=d["year"], y=d["afford_ratio"],
        mode="lines+markers",
        name=region,
        line=dict(color=REGION_COLORS[region], width=2.5),
        marker=dict(size=5),
        hovertemplate="%{x}: %{y:.2f}x<extra>" + region + "</extra>",
    ))

fig.add_hline(y=1.0, line_dash="dash", line_color="black", line_width=1.5,
              annotation_text="2007 baseline (affordable = 1.0)",
              annotation_position="bottom right")
fig.update_layout(
    title="Affordability Ratio: Housing Price Growth / Real Income Growth (2007 = 1.0)",
    xaxis_title="Year",
    yaxis_title="Ratio (1.0 = same affordability as 2007)",
    template="plotly_white",
    height=490,
    hovermode="x unified",
    legend=dict(orientation="v", x=1.01),
)
fig.show()


## Visualisation 6 – Heatmap: Affordability Ratio by Region and Year
A compact overview of which regions became unaffordable and when. Red = housing has outpaced income significantly; green = roughly in line with 2007 affordability.

In [29]:

df_aff_pivot = df_aff.pivot(index="region", columns="year", values="afford_ratio")
df_aff_pivot = df_aff_pivot.loc[df_aff_pivot.iloc[:, -1].sort_values(ascending=False).index]

fig = px.imshow(
    df_aff_pivot,
    color_continuous_scale=["#2ecc71", "#f1c40f", "#e74c3c"],
    zmin=0.8, zmax=df_aff_pivot.values.max(),
    labels={"color": "Ratio", "x": "Year", "y": "Region"},
    title="Heatmap – Affordability Ratio by Region (2007 = 1.0, red = less affordable)",
    aspect="auto",
    text_auto=".2f",
)
fig.update_layout(template="plotly_white", height=400,
                  coloraxis_colorbar_title="Ratio")
fig.show()


## Visualisation 7 – Side-by-Side: National Housing Price vs Real Income (Filled Area)
Overlapping filled areas for the national average housing price index and real income index. The growing gap between the two curves is the affordability shortfall at the national level.

In [30]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_inc["year"], y=df_inc["nom_wage_2007"],
    mode="lines", name="Nominal wage index",
    fill=None,
    line=dict(color="#888888", width=1.5, dash="dot"),
))
fig.add_trace(go.Scatter(
    x=df_inc["year"], y=df_inc["real_income_2007"],
    mode="lines", name="Real income per capita",
    fill="tonexty",
    line=dict(color="#2a9d8f", width=2.5),
    fillcolor="rgba(42,157,143,0.15)",
))
fig.add_trace(go.Scatter(
    x=df_nat_price["year"], y=df_nat_price["nat_price_idx"],
    mode="lines+markers", name="National avg housing price",
    fill="tonexty",
    line=dict(color="#e63946", width=2.5),
    fillcolor="rgba(230,57,70,0.15)",
    marker=dict(size=5),
))

fig.add_hline(y=100, line_dash="dot", line_color="lightgrey", line_width=1)
fig.update_layout(
    title="National Housing Price vs Real Income – Growth Since 2007 (2007 = 100)",
    xaxis_title="Year",
    yaxis_title="Index (2007 = 100)",
    template="plotly_white",
    height=490,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.18),
)
fig.show()
